In [1]:
import os
import json
import pandas as pd
import vertexai
from vertexai.generative_models import GenerativeModel, Part
from concurrent.futures import ThreadPoolExecutor

Task was destroyed but it is pending!
task: <Task pending name='Task-59' coro=<_async_in_context.<locals>.run_in_context_pre311() running at /opt/conda/lib/python3.10/site-packages/ipykernel/utils.py:76> cb=[ZMQStream._run_callback.<locals>._log_error() at /opt/conda/lib/python3.10/site-packages/zmq/eventloop/zmqstream.py:563]>
/opt/conda/lib/python3.10/asyncio/base_events.py:674: RuntimeWarning: coroutine '_async_in_context.<locals>.run_in_context_pre311' was never awaited
  self._ready.clear()
/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.cloud.aiplatform_v1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.aiplatform_v1 past that date.
  warnings.warn(message, FutureWarning)
/opt/conda/lib/python3.10/site-packages/go

In [2]:
pip install pyinaturalist

Note: you may need to restart the kernel to use updated packages.


In [3]:
import io
import os
import csv
import random
import pandas as pd
from PIL import Image, ImageEnhance
from concurrent.futures import ThreadPoolExecutor

# Scrape Relevant Photos from iNaturalist

In [4]:
import time
import requests
from pyinaturalist import get_observations

import csv
from datetime import datetime

In [5]:
ls

'Gemini Confidence Testing.ipynb'    animal_dataset_images/
 _gemini_2.5_flash/                  charts.ipynb
 _gemini_2.5_flash_cdrl/             charts_all_images/
 _gemini_2.5_flash_ssot/             dataset_metadata.csv
 all_images_distortion_results.csv   gemini_beached_analysis.csv


In [6]:
# want to scrape these species from iNaturalist
# add birds, beavers, etc

animal_scrape_dict = {
  'California Sea Lion' : 'Zalophus californianus',
  #'Northern Elephant Seal': 'Mirounga angustirostris',
  'Harbor Seal' : 'Phoca vitulina',
  'Common Dolphin' : 'Delphinus delphis',
  #'Bottlenose Dolphin' : 'Tursiops truncatus',
  'American Beaver' : 'Castor canadensis',
  'Ruby-throated Hummingbird' : 'Archilochus colubris',
  'Canada Goose' : 'Branta canadensis',
  'Domestic Dog' : 'Canis familiaris',
  'Loggerhead Sea Turtle' : 'Caretta caretta',
  'Mallard' : 'Anas platyrhynchos',
  'Eastern Gray Squirrel' : 'Sciurus carolinensis'

}



In [7]:
IMAGE_FOLDER = "animal_dataset_images"
CSV_FILE_NAME = "dataset_metadata.csv"
os.makedirs(IMAGE_FOLDER, exist_ok=True)

In [11]:
#SEARCH_TERMS = ['stranded', 'rescue', 'beach', 'land'] # Add more as needed
MAX_PHOTOS_PER_SPECIES = 40

os.makedirs(IMAGE_FOLDER, exist_ok=True)

fieldnames = [
    'animal_key', 'animal_value', 'observation_id', 
    'latitude', 'longitude', 'date', 'time', 
    'description', 'image_filename'
]

# Initialize and overwrite the CSV
with open(CSV_FILE_NAME, mode='w', newline='', encoding='utf-8') as main_csv:
    writer = csv.DictWriter(main_csv, fieldnames=fieldnames)
    writer.writeheader()

# Global tracker to avoid duplicates across different species or terms
# If an ID is in this set, we skip it entirely
downloaded_ids = set()

# --- Main Scrape Loop ---
for key, value in animal_scrape_dict.items():
    print(f"\n{'='*40}")
    print(f"Target Species: {value} ({key})")
    print(f"{'='*40}")
    
    photos_per_species_count = 0
    
    #for term in SEARCH_TERMS:
        # Stop searching if we already hit the limit for this species
    if photos_per_species_count >= MAX_PHOTOS_PER_SPECIES:
        break
        
    #print(f"Searching for term: '{term}'...")
    
    observations = get_observations(
        #q=term,
        search_on='description',
        taxon_name=value,
        photos=True,
        quality_grade='research',
        per_page=200,
        order_by='created_at',
        order='desc',
        d1="2025-01-01", #knowledge cutoff for gemini 2.5 and 3
    )

    results = observations.get('results', [])
    
    # Open in append mode to save results incrementally
    with open(CSV_FILE_NAME, mode='a', newline='', encoding='utf-8') as main_csv:
        writer = csv.DictWriter(main_csv, fieldnames=fieldnames)

        for obs in results:
            # BREAK if we hit 200 for this specific animal
            if photos_per_species_count >= MAX_PHOTOS_PER_SPECIES:
                break
            
            obs_id = obs['id']
            
            # AVOID DUPLICATES
            if obs_id in downloaded_ids:
                continue

            # 1. Location Parsing
            location = obs.get('location')
            lat, lon = (None, None)
            if location:
                if isinstance(location, list) and len(location) == 2:
                    lat, lon = location[0], location[1]
                elif isinstance(location, str):
                    parts = location.split(',')
                    if len(parts) == 2:
                        lat, lon = parts[0], parts[1]

            # 2. Date & Time Parsing
            time_raw = obs.get('created_at_details')
            obs_date, obs_time = ("Unknown", "Unknown")
            if time_raw:
                obs_date = time_raw.get('date', "Unknown")
                obs_time = time_raw.get('hour', "Unknown")

            # 3. Description Cleaning
            desc = (obs.get('description') or "").replace('\n', ' ').replace('\r', ' ')

            # 4. Image Download
            file_name = f"{obs_id}.jpg"
            file_path = os.path.join(IMAGE_FOLDER, file_name)
            
            success = False
            if obs['photos']:
                img_url = obs['photos'][0]['url'].replace('square', 'large')
                try:
                    img_data = requests.get(img_url, timeout=10).content
                    with open(file_path, 'wb') as f:
                        f.write(img_data)
                    success = True
                except Exception as e:
                    print(f"  Error downloading {obs_id}: {e}")

            # 5. Save and Increment
            if success:
                writer.writerow({
                    'animal_key': key,
                    'animal_value': value,
                    'observation_id': obs_id,
                    'latitude': lat,
                    'longitude': lon,
                    'date': obs_date,
                    'time': obs_time,
                    'description': desc,
                    'image_filename': file_name
                })
                downloaded_ids.add(obs_id) # Mark as downloaded
                photos_per_species_count += 1

    # Small cooldown between keyword searches for the same species
    time.sleep(2)

    # print(f"Finished {value}. Total images for this species: {photos_per_species_count}")
    # print("Waiting 15 seconds for API cooldown...")
    # time.sleep(15)

print("\nFull dataset complete!")


Target Species: Zalophus californianus (California Sea Lion)

Target Species: Phoca vitulina (Harbor Seal)

Target Species: Delphinus delphis (Common Dolphin)

Target Species: Castor canadensis (American Beaver)

Target Species: Archilochus colubris (Ruby-throated Hummingbird)

Target Species: Branta canadensis (Canada Goose)

Target Species: Canis familiaris (Domestic Dog)

Target Species: Caretta caretta (Loggerhead Sea Turtle)

Target Species: Anas platyrhynchos (Mallard)

Target Species: Sciurus carolinensis (Eastern Gray Squirrel)

Full dataset complete!


In [12]:
ls

'Gemini Confidence Testing.ipynb'   dataset_metadata.csv
 animal_dataset_images/


In [14]:
len(os.listdir('./animal_dataset_images/'))

401

In [13]:

# --- Configuration ---
PROJECT_ID = 'spring-2026-lifejacket'
LOCATION = 'us-central1'
IMAGE_FOLDER = "animal_dataset_images"
INPUT_CSV = "dataset_metadata.csv"
OUTPUT_CSV = "gemini_beached_analysis.csv"

vertexai.init(project=PROJECT_ID, location=LOCATION)

# Gemini confidence testing

## Confidence ranges for a given photo

In [6]:
os.getcwd()

'/home/jupyter/Gemini Confidence Testing'

In [ ]:
#beached images

#dolphin: 361231332
#turtle: 362234822, 362502571

# 362487982 362491926

In [ ]:
## find one photo for each species, and give gemini the same prompt and photo. 
## see what the distribution of confidence scores are
## try color jittering and rotation to see if that does anything to confidence as well

#california sea lion: 	observation_id: 362499770
#harbor seal:	observation_id: 362345207
#turtle: observation_id: 362423278  
#common dolphin: 361231332
#canada goose: 362736713
#domestic dog: 362236437 361933579


In [12]:
MODEL = 'gemini-2.5-flash'
model = GenerativeModel(MODEL)

/opt/conda/lib/python3.10/site-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


In [10]:
#test using dolphin (361231332) and 
TARGET_IDS = [362499770, 362345207, 362423278, 361654652, 362736713, 362236437]

TRIALS_PER_LEVEL = 50  # 50 trials * 5 distortion levels = 100 API requests per ID
MAX_WORKERS = 15       # Concurrent threads hitting the API simultaneously

In [11]:
DISTORTION_CONFIGS = {
    # Level 0: Perfect baseline (100% legibility)
    0: {"brightness": (1.0, 1.0), "contrast": (1.0, 1.0), "color": (1.0, 1.0), "sharpness": (1.0, 1.0), "rotate": False},
    
    # Level 1: Clean but slightly off (Mild exposure shift, clear details remain)
    1: {"brightness": (0.80, 1.20), "contrast": (0.85, 1.15), "color": (0.75, 1.25), "sharpness": (0.70, 1.30), "rotate": False},
    
    # Level 2: Noticeably muddy & blurred (Forced structural soften, losing fine textures)
    2: {"brightness": (0.60, 0.79), "contrast": (0.65, 0.84), "color": (0.50, 0.74), "sharpness": (0.20, 0.50), "rotate": False},
    
    # Level 3: Severe contrast loss & rotation (Highly faded, washed out color, orientation flipped)
    3: {"brightness": (0.40, 0.59), "contrast": (0.35, 0.64), "color": (0.15, 0.49), "sharpness": (0.01, 0.15), "rotate": True},
    
    # Level 4: Extreme degradation (Heavy silhouette or extreme overexposure ghosting)
    4: {"brightness": (0.20, 0.39), "contrast": (0.20, 0.34), "color": (0.01, 0.14), "sharpness": (0.0, 0.0), "rotate": True},
    
    # Level 5: The "Ghostly Fog" Threshold (Fully visible details are erased into a flat, blurry gray haze)
    5: {"brightness": (0.45, 0.55), "contrast": (0.05, 0.12), "color": (0.0, 0.0), "sharpness": (0.0, 0.0), "rotate": True}
}
OUTPUT_FIELDS = [
    'trial', 'distortion_level', 'observation_id', 'actual_name', 'actual_species',
    'top1_common_name', 'top1_genus', 'top1_species', 'top1_prob',
    'top2_common_name', 'top2_genus', 'top2_species', 'top2_prob',
]

In [22]:
##OLD

def get_regular_prompt(metadata_context):
    return f"""You are an expert wildlife species identification model. 
OBSERVATION DATA: {metadata_context}

TASK:
Identify the top 2 likely species. 

PROBABILITY RULES:
- Probabilities must be integers from 0-100
- Probabilities DO NOT need to sum to 100
- Lower confidence should produce lower probabilities
- If multiple species are similarly plausible, reflect uncertainty honestly
- Avoid overconfident predictions
- A probability of 90 should mean you are 90% certain you are correct based on CLEAR visual traits.

OUTPUT FORMAT:
Exactly this format with pipe separators and NOTHING else:
Top1 Common Name | Top1 Genus | Top1 Species | Top1 Probability | Top2 Common Name | Top2 Genus | Top2 Species | Top2 Probability

EXAMPLE:
Harbor Seal | Phoca | Phoca vitulina | 95 | Grey Seal | Halichoerus | Halichoerus grypus | 12"""

def get_cdrl_prompt(metadata_context):
    return f"""SYSTEM:
You are an expert wildlife species identification model. 
You must link perception directly to accuracy.

OBSERVATION DATA:
{metadata_context}

PHASE 1: MULTI-MODULE INTERNAL EVALUATION
1. VISUAL SELF-CHECK: Evaluate the image's perceptual clarity. Identify specific tokens (ear shape, tail, texture). Identify 'Perceptual Barriers' (blur, rotation, fading).
2. SELF-CONSISTENCY AUDIT: 
   - Path A (Visual): What species does the image suggest?
   - Path B (Contextual): What species does the metadata suggest?
   - Compare: If Path A and Path B conflict, or clarity is low, you MUST lower your confidence.
3. SELF-REFLECTION: Act as a critic. Identify the most likely 'Look-alike' species for this specific degradation.

PHASE 2: CALIBRATED OUTPUT
- Your probability score measures agreement between Visual Self-Check and Self-Consistency.
- High Agreement + High Clarity = High Probability (80-100).
- Low Agreement OR Low Clarity = Low Probability (under 40).

OUTPUT FORMAT:
Exactly this format with pipe separators and NOTHING else:
Top1 Common Name | Top1 Genus | Top1 Species | Top1 Probability | Top2 Common Name | Top2 Genus | Top2 Species | Top2 Probability

EXAMPLE:
Harbor Seal | Phoca | Phoca vitulina | 95 | Grey Seal | Halichoerus | Halichoerus grypus | 12"""

def get_ssot_prompt(metadata_context):
    return f"""SYSTEM:
You are an expert wildlife species identification model.
Your task is to identify animals from images using:
1. visual appearance,
2. observation metadata,
3. geographic plausibility,
4. seasonal and temporal likelihood.

OBSERVATION DATA:
{metadata_context}

SSOT RANDOMNESS STEP:
Before deciding, internally generate a short random 8-character seed string.
Use the seed to introduce controlled diversity when multiple species are visually plausible.
The seed should ONLY affect tie-breaking or uncertain cases.
Do NOT reveal the seed.

REASONING PROCESS:
Internally evaluate:
- body shape
- fur/feather/skin texture
- coloration and markings
- pose and silhouette
- habitat consistency
- time/date plausibility
- species prevalence in region
- visual ambiguity

Then internally rank candidate species by confidence.

TASK:
Return the TOP 2 most likely species predictions.

PROBABILITY RULES:
- Probabilities must be integers from 0-100
- Probabilities DO NOT need to sum to 100
- Lower confidence should produce lower probabilities
- If multiple species are similarly plausible, reflect uncertainty honestly
- Avoid overconfident predictions

OUTPUT FORMAT:
Exactly this format with pipe separators and NOTHING else:

Top1 Common Name | Top1 Genus | Top1 Species | Top1 Probability | Top2 Common Name | Top2 Genus | Top2 Species | Top2 Probability

EXAMPLE:
Harbor Seal | Phoca | Phoca vitulina | 95 | Grey Seal | Halichoerus | Halichoerus grypus | 12
"""

In [14]:

def process_single_row(task):
    trial_id, distortion_level, is_first_of_level, row, csv_filename, target_img_dir = task
    img_path = os.path.join(IMAGE_FOLDER, row['image_filename'])
    
    if not os.path.exists(img_path):
        return None

    metadata_context = (
        f"Location: ({row['latitude']}, {row['longitude']}), "
        f"Date/Time: {row['date']} at {row['time']} Military time"
    )

    try:
        cfg = DISTORTION_CONFIGS[distortion_level]
        
        with Image.open(img_path) as img:
            img = img.convert("RGB")
            
            if cfg["rotate"]:
                rotation_choice = random.choice([Image.ROTATE_90, Image.ROTATE_180, Image.ROTATE_270])
                if rotation_choice:
                    img = img.transpose(rotation_choice)
            
            img = ImageEnhance.Brightness(img).enhance(random.uniform(*cfg["brightness"]))
            img = ImageEnhance.Contrast(img).enhance(random.uniform(*cfg["contrast"]))
            img = ImageEnhance.Color(img).enhance(random.uniform(*cfg["color"]))
            img = ImageEnhance.Sharpness(img).enhance(random.uniform(*cfg["sharpness"]))
            
            # Save transformed image to bytes for API delivery
            img_byte_arr = io.BytesIO()
            img.save(img_byte_arr, format='JPEG', quality=85)
            image_bytes = img_byte_arr.getvalue()
            
            # === SAVES ONLY 1 SAMPLE PER DISTORTION LEVEL ===
            if is_first_of_level:
                label = "normal" if distortion_level == 0 else f"level_{distortion_level}"
                preview_filename = f"sample_{label}.jpg"
                preview_path = os.path.join(target_img_dir, preview_filename)
                img.save(preview_path, format='JPEG', quality=85)

        image_part = Part.from_data(data=image_bytes, mime_type="image/jpeg")

        #get diff prompts here:
        #prompt = get_regular_prompt(metadata_context=metadata_context)
        prompt = get_ssot_prompt(metadata_context=metadata_context)
        #prompt = get_cdrl_prompt(metadata_context=metadata_context)
        
        response = model.generate_content([prompt, image_part])
        raw_res = response.text.strip()
    except Exception as e:
        raw_res = f"Error | Error | Error | 0 | Error | Error | Error | 0"

    parts = [p.strip() for p in raw_res.split('|')]
    while len(parts) < 8:
        parts.append("N/A")

    result = {
        'trial': trial_id,
        'distortion_level': distortion_level,
        'observation_id': row['observation_id'],
        'actual_name': row['animal_key'],
        'actual_species': row['animal_value'],
        'top1_common_name': parts[0],
        'top1_genus': parts[1],
        'top1_species': parts[2],
        'top1_prob': parts[3],
        'top2_common_name': parts[4],
        'top2_genus': parts[5],
        'top2_species': parts[6],
        'top2_prob': parts[7],
    }

    # Thread-safe appending directly to CSV
    with open(csv_filename, mode='a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=OUTPUT_FIELDS)
        writer.writerow(result)

    return result



In [15]:
INPUT_CSV

'dataset_metadata.csv'

In [120]:
df = pd.read_csv(INPUT_CSV)

In [121]:
df[df['observation_id'] == 362499770]

,animal_key,animal_value,observation_id,latitude,longitude,date,time,description,image_filename
21,California Sea Lion,Zalophus californianus,362499770,48.115386,-122.751008,2026-05-17,12,NaN,362499770.jpg


In [16]:
import vertexai
from vertexai.generative_models import GenerativeModel, Part
from concurrent.futures import ThreadPoolExecutor
import traceback

In [123]:
df = pd.read_csv(INPUT_CSV)

BASE_PREVIEW_DIR = "distortion_images"
os.makedirs(BASE_PREVIEW_DIR, exist_ok=True)

for idx, obs_id in enumerate(TARGET_IDS, start=1):
    print(f"--- [Batch {idx}/{len(TARGET_IDS)}] Profiling Observation ID: {obs_id} ---")
    
    matching_rows = df[df['observation_id'] == obs_id]
    if matching_rows.empty:
        print(f"observation_id {obs_id} not indexed. Skipping.")
        continue
        
    target_row = matching_rows.iloc[0]
    
    dynamic_output_csv = f"{obs_id}_distribution_results.csv"
    target_img_dir = os.path.join(BASE_PREVIEW_DIR, f"images_{obs_id}")
    os.makedirs(target_img_dir, exist_ok=True)
    
    with open(dynamic_output_csv, mode='w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=OUTPUT_FIELDS)
        writer.writeheader()
        
    # Compile parameter loop spanning Level 0 through Level 5 (6 loops total)
    tasks = []
    global_trial_count = 1
    for level in range(0, 6):
        for inner_idx in range(TRIALS_PER_LEVEL):
            # If inner_idx is 0, this is the very first trial running for this specific level tier
            is_first_of_level = (inner_idx == 0)
            tasks.append((global_trial_count, level, is_first_of_level, target_row, dynamic_output_csv, target_img_dir))
            global_trial_count += 1
            
    print(f"Dispatching {len(tasks)} parallel requests (Isolated to 6 reference images saved)...")
    
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        _ = list(executor.map(process_single_row, tasks))
        
    print(f"Complete. Output CSV: {dynamic_output_csv}")
    print(f"Saved 6 reference image files inside: {target_img_dir}\n")

print("All batch profiles executed. System processing run fully finished.")

--- [Batch 1/6] Profiling Observation ID: 362499770 ---
Dispatching 300 parallel requests (Isolated to 6 reference images saved)...
Complete. Output CSV: 362499770_distribution_results.csv
Saved 6 reference image files inside: distortion_images/images_362499770

--- [Batch 2/6] Profiling Observation ID: 362345207 ---
Dispatching 300 parallel requests (Isolated to 6 reference images saved)...
Complete. Output CSV: 362345207_distribution_results.csv
Saved 6 reference image files inside: distortion_images/images_362345207

--- [Batch 3/6] Profiling Observation ID: 362423278 ---
Dispatching 300 parallel requests (Isolated to 6 reference images saved)...
Complete. Output CSV: 362423278_distribution_results.csv
Saved 6 reference image files inside: distortion_images/images_362423278

--- [Batch 4/6] Profiling Observation ID: 361654652 ---
Dispatching 300 parallel requests (Isolated to 6 reference images saved)...
Complete. Output CSV: 361654652_distribution_results.csv
Saved 6 reference imag

# All Images

In [23]:
def process_single_row(task):
    trial_id, distortion_level, row = task
    img_path = os.path.join(IMAGE_FOLDER, row['image_filename'])
    
    if not os.path.exists(img_path):
        return None

    metadata_context = (
        f"Location: ({row['latitude']}, {row['longitude']}), "
        f"Date/Time: {row['date']} at {row['time']} Military time"
    )

    try:
        cfg = DISTORTION_CONFIGS[distortion_level]
        
        with Image.open(img_path) as img:
            img = img.convert("RGB")
            
            if cfg["rotate"]:
                rotation_choice = random.choice([Image.ROTATE_90, Image.ROTATE_180, Image.ROTATE_270])
                img = img.transpose(rotation_choice)
            
            img = ImageEnhance.Brightness(img).enhance(random.uniform(*cfg["brightness"]))
            img = ImageEnhance.Contrast(img).enhance(random.uniform(*cfg["contrast"]))
            img = ImageEnhance.Color(img).enhance(random.uniform(*cfg["color"]))
            img = ImageEnhance.Sharpness(img).enhance(random.uniform(*cfg["sharpness"]))
            
            img_byte_arr = io.BytesIO()
            img.save(img_byte_arr, format='JPEG', quality=85)
            image_bytes = img_byte_arr.getvalue()

        image_part = Part.from_data(data=image_bytes, mime_type="image/jpeg")
        
        prompt = get_cdrl_prompt(metadata_context=metadata_context)
        #print(prompt)
        
        response = model.generate_content([prompt, image_part])
        raw_res = response.text.strip()
    except Exception:
        raw_res = "Error | Error | Error | 0 | Error | Error | Error | 0"
        traceback.print_exc()

    parts = [p.strip() for p in raw_res.split('|')]
    while len(parts) < 8:
        parts.append("N/A")

    result = {
        'trial': trial_id,
        'distortion_level': distortion_level,
        'observation_id': row['observation_id'],
        'actual_name': row['animal_key'],
        'actual_species': row['animal_value'],
        'top1_common_name': parts[0],
        'top1_genus': parts[1],
        'top1_species': parts[2],
        'top1_prob': parts[3],
        'top2_common_name': parts[4],
        'top2_genus': parts[5],
        'top2_species': parts[6],
        'top2_prob': parts[7],
    }

    # Thread-safe appending to the single output file
    with open(OUTPUT_CSV, mode='a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=OUTPUT_FIELDS)
        writer.writerow(result)

    return result


In [24]:
df = pd.read_csv(INPUT_CSV)

In [25]:
df.head()

,animal_key,animal_value,observation_id,latitude,longitude,date,time,description,image_filename
0,California Sea Lion,Zalophus californianus,362648354,37.810678,-122.411247,2026-05-17,22,NaN,362648354.jpg
1,California Sea Lion,Zalophus californianus,362624044,37.810730,-122.411014,2026-05-17,19,NaN,362624044.jpg
2,California Sea Lion,Zalophus californianus,362623742,37.810730,-122.411222,2026-05-17,19,NaN,362623742.jpg
3,California Sea Lion,Zalophus californianus,362623564,37.810526,-122.411103,2026-05-17,19,NaN,362623564.jpg
4,California Sea Lion,Zalophus californianus,362623387,37.809647,-122.410436,2026-05-17,19,NaN,362623387.jpg


In [26]:
OUTPUT_CSV = 'all_images_distortion_results.csv'
TRIALS_PER_LEVEL=2
MAX_WORKERS=15
IMAGE_FOLDER = "animal_dataset_images"

In [ ]:


with open(OUTPUT_CSV, mode='w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=OUTPUT_FIELDS)
    writer.writeheader()

df = pd.read_csv(INPUT_CSV)
tasks = []
global_trial_id = 1

print(f"Preparing tasks for {len(df)} images...")

# 2. Build task list: Level 0-5 for every single row in the CSV
for _, row in df.iterrows():
    for level in range(0, 6):
        for _ in range(TRIALS_PER_LEVEL):
            tasks.append((global_trial_id, level, row))
            global_trial_id += 1

print(f"Starting processing: {len(tasks)} total requests...")

# 3. Process in parallel
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    _ = list(executor.map(process_single_row, tasks))

print(f"All processing finished. Results saved to: {OUTPUT_CSV}")


Preparing tasks for 400 images...
Starting processing: 4800 total requests...
